## 🎯 Learning Objectives
* Understand the concept of nested chats and composable conversation patterns in AutoGen.
* Learn how to design and implement agents that can initiate and manage sub-conversations.
* Identify scenarios where nested chats improve modularity, scalability, and task decomposition in multi-agent systems.
* Analyze the performance implications and best practices for using nested chat patterns.


## Nested Chats and Composable Conversation Patterns in AutoGen

In the evolving landscape of AI agents, managing complexity and enabling specialized collaboration is paramount. As we move towards more sophisticated agentic systems, the need for agents to not just converse, but to *orchestrate* and *delegate* tasks to sub-teams or specialized agents becomes critical. This is where **nested chats** and **composable conversation patterns** in AutoGen shine.

Imagine a large software development project. A Project Manager doesn't personally write every line of code or test every feature. Instead, they delegate tasks to a development team, a QA team, or a design team. Each of these teams might then have their own internal discussions and workflows to complete their assigned sub-task, eventually reporting back to the Project Manager with their findings or completed work.

This real-world analogy perfectly maps to nested chats in AutoGen. A **main conversation** (like the Project Manager's overarching project chat) can contain agents that, upon identifying a specific sub-task, initiate a **sub-conversation** (a nested chat) involving a different set of specialized agents. Once the sub-conversation concludes, its outcome or summary is then fed back into the main conversation, allowing the primary orchestrating agent to continue its work.

### Why are Nested Chats Important?

1.  **Modularity and Specialization**: Just like functions in programming, nested chats allow you to encapsulate specific workflows. A 'code generation' sub-chat can involve a Software Engineer and a Code Reviewer, while a 'data analysis' sub-chat might involve a Data Scientist and a Statistician. This promotes clear roles and responsibilities.
2.  **Complexity Management**: Breaking down a large, complex problem into smaller, manageable sub-problems makes the overall system easier to design, debug, and maintain. Each nested chat focuses on a narrow scope.
3.  **Reusability**: A well-defined nested chat pattern (e.g., a 'bug fixing' team) can be reused across different main conversations or projects, leading to more efficient agent development.
4.  **Dynamic Orchestration**: Agents can dynamically decide when and how to initiate a sub-conversation based on the current task or context, leading to more adaptive and intelligent systems.

### How AutoGen Facilitates This (2026 Perspective)

AutoGen's architecture, particularly with its `GroupChat` and `GroupChatManager` components, is inherently designed for flexible multi-agent interactions. While earlier versions required more manual orchestration, modern AutoGen (as of 2026) provides more streamlined APIs and patterns for agents to programmatically initiate and manage sub-chats. This often involves:

*   **Custom Agent Logic**: An agent's `generate_reply` method can be augmented or overridden to include logic that detects the need for a sub-conversation.
*   **Dynamic `GroupChat` Creation**: Agents can instantiate new `GroupChat` and `GroupChatManager` instances on the fly, populating them with relevant agents for the sub-task.
*   **Context Passing**: The orchestrating agent passes the necessary context to the nested chat and then processes the summary or output from the nested chat to continue the main conversation.

This pattern allows for highly composable and hierarchical agentic workflows, moving beyond simple round-robin conversations to truly intelligent, delegated problem-solving.


In [ ]:
import autogen
import os

# --- Configuration --- 
# Ensure you have your OpenAI API key set as an environment variable
# For example: os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# Using a modern, capable model for 2026
# Replace with your preferred LLM endpoint and model if not using OpenAI
config_list = [
    {
        "model": "gpt-4o-2024-05-13", # Or a similar powerful model like Claude 3.5 Sonnet, Gemini 1.5 Pro
        "api_key": os.environ.get("OPENAI_API_KEY")
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0.7,
    "timeout": 120
}

# --- Define Agents for the Main Conversation --- 

# User Proxy Agent: Represents the human user initiating the main task
user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER", # Set to "ALWAYS" or "TERMINATE" for human interaction
    max_consecutive_auto_reply=10, # Allow up to 10 auto-replies before human input (if enabled)
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").upper(),
    code_execution_config=False # No code execution for this example
)

# Project Manager Agent: Orchestrates tasks, delegates to sub-teams
# This agent will contain the logic to initiate a nested chat
project_manager = autogen.ConversableAgent(
    name="ProjectManager",
    system_message=(
        "You are a seasoned Project Manager. Your primary role is to oversee project tasks, "
        "delegate specific implementation and quality assurance tasks to the Dev/QA team, "
        "and report their progress back to the User. When asked to 'implement a feature', "
        "you will initiate a nested conversation with the Software Engineer and QA Engineer. "
        "Summarize the outcome of the nested chat for the User. "
        "Always end your final response with 'TERMINATE' when the overall task is complete."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").upper()
)

# --- Custom Reply Function for ProjectManager to Initiate Nested Chat ---

def initiate_nested_dev_qa_chat(recipient, messages, sender, config):
    """
    Custom reply function for the ProjectManager to initiate a nested chat
    with Software Engineer and QA Engineer.
    """
    last_message_content = messages[-1]["content"]

    # Check for a trigger phrase to initiate the nested chat
    if "implement feature" in last_message_content.lower() and "user profile" in last_message_content.lower():
        print("\n--- ProjectManager: Delegating 'user profile' feature implementation to Dev/QA team (Nested Chat) ---")

        # Define agents specifically for the nested chat
        # These can be the same agent types but with specific roles for the sub-task
        software_engineer_nested = autogen.ConversableAgent(
            name="SoftwareEngineer_Nested",
            system_message=(
                "You are a skilled Software Engineer. Your task is to implement the requested feature, "
                "including data models, API endpoints, and comprehensive unit tests. "
                "Collaborate with the QA Engineer to ensure code quality and test coverage. "
                "Once implemented and tested, report 'Feature implemented and tested.'"
            ),
            llm_config=config["llm_config"],
            human_input_mode="NEVER",
            is_termination_msg=lambda x: "feature implemented and tested" in x.get("content", "").lower()
        )

        qa_engineer_nested = autogen.ConversableAgent(
            name="QAEngineer_Nested",
            system_message=(
                "You are a meticulous QA Engineer. Your role is to review the code and unit tests "
                "provided by the Software Engineer. Provide constructive feedback, identify bugs, "
                "and ensure the feature meets quality standards. "
                "Report 'Tests passed and code satisfactory.' when done."
            ),
            llm_config=config["llm_config"],
            human_input_mode="NEVER",
            is_termination_msg=lambda x: "tests passed and code satisfactory" in x.get("content", "").lower()
        )

        # Create a GroupChat for the nested conversation
        nested_groupchat = autogen.GroupChat(
            agents=[software_engineer_nested, qa_engineer_nested],
            messages=[],
            max_round=10, # Max rounds for the nested chat
            speaker_selection_method="auto" # Auto-select speaker based on LLM
        )
        nested_manager = autogen.GroupChatManager(groupchat=nested_groupchat, llm_config=config["llm_config"])

        # Initiate the nested chat with a specific task message
        nested_manager.initiate_chat(
            nested_groupchat,
            message=(
                f"Project Manager has assigned: {last_message_content}. "
                "Software Engineer, please start by outlining the implementation plan. "
                "QA Engineer, prepare to review."
            ),
            sender=software_engineer_nested # The first speaker in the nested chat
        )

        print("\n--- Nested Chat Ended. ProjectManager summarizing for main chat. ---")

        # Extract the summary from the nested chat's last message
        # In a real system, you might have a dedicated summarizer agent or a more structured output.
        nested_chat_summary = f"The Dev/QA team has completed the task. " \
                              f"Their final report: {nested_groupchat.messages[-1]['content']}"
        return nested_chat_summary
    else:
        # If no nested chat is needed, let the LLM generate a default reply
        return autogen.ConversableAgent.generate_reply(recipient, messages, sender, config)

# Register the custom reply function with the ProjectManager agent
project_manager.register_reply(
    [autogen.Agent, None], # This agent can reply to any agent or no specific agent
    reply_func=initiate_nested_dev_qa_chat,
    config={"llm_config": llm_config} # Pass llm_config to the custom function
)

# --- Main Conversation Setup --- 

# Define the agents for the main conversation
main_agents = [user_proxy, project_manager]

# Create the main GroupChat
main_groupchat = autogen.GroupChat(
    agents=main_agents,
    messages=[],
    max_round=15, # Max rounds for the main chat
    speaker_selection_method="auto" # Auto-select speaker based on LLM
)

# Create the main GroupChatManager
main_manager = autogen.GroupChatManager(groupchat=main_groupchat, llm_config=llm_config)

# --- Initiate the Main Conversation ---
print("\n--- Initiating Main Conversation ---")
user_proxy.initiate_chat(
    main_manager,
    message="I need a new 'user profile' feature implemented. Project Manager, please handle this."
)
print("\n--- Main Conversation Ended ---")


### Interpreting the Output and Use Cases

When you run the code, you'll observe a clear flow:

1.  **Main Chat Initiation**: The `User` agent sends a request to the `ProjectManager`.
2.  **Delegation and Nested Chat Trigger**: The `ProjectManager` agent, using its custom reply logic, detects the keyword "implement feature" and "user profile". Instead of directly replying, it prints a message indicating it's delegating and then programmatically sets up and initiates a *new*, independent `GroupChat` (the nested chat) involving the `SoftwareEngineer_Nested` and `QAEngineer_Nested`.
3.  **Nested Chat Execution**: You'll see the conversation unfold between `SoftwareEngineer_Nested` and `QAEngineer_Nested`. They will discuss the implementation, testing, and review process. This sub-conversation runs to completion based on its own `max_round` and termination conditions.
4.  **Nested Chat Summary and Main Chat Continuation**: Once the nested chat terminates, the `ProjectManager` extracts the final message or a summary from the nested chat's history. It then uses this summary to formulate its reply back to the main conversation, effectively reporting the sub-team's progress or completion. The main chat then continues, potentially with further instructions or a `TERMINATE` message.

This demonstrates a powerful **composable pattern**: an agent in a higher-level conversation can dynamically spawn and manage lower-level, specialized conversations, integrating their outcomes back into the main flow.

### Performance Trade-offs

While incredibly powerful, nested chats introduce certain considerations:

*   **Increased LLM Calls**: Each nested chat involves its own series of LLM calls. A deeply nested or highly iterative sub-conversation can significantly increase token usage and API costs.
*   **Latency**: The total time to complete a task will be the sum of the main chat's rounds plus the nested chat's rounds. Deep nesting can lead to higher overall latency.
*   **Context Management**: While AutoGen handles context within each chat, the orchestrating agent needs to effectively summarize and pass relevant context to and from nested chats to avoid information loss or redundancy. Poor summarization can lead to 


### Resources

*   **AutoGen Official Documentation**: The primary source for understanding `GroupChat`, `GroupChatManager`, and `ConversableAgent` capabilities. Look for advanced examples on custom agent behaviors and chat orchestration.
    *   [AutoGen GitHub Repository](https://github.com/microsoft/autogen)
    *   [AutoGen Documentation](https://microsoft.github.io/autogen/docs/)
*   **Microsoft Research Papers on AutoGen**: Explore the foundational papers that detail the multi-agent framework and its design principles.
    *   [AutoGen: Enabling Next-Gen LLM Applications with Multi-Agent Conversation Framework](https://arxiv.org/abs/2308.08155)
*   **OpenAI API Documentation**: For understanding the underlying LLM capabilities and best practices for prompt engineering and function calling, which can enhance agent intelligence.
    *   [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
*   **Hugging Face Models**: If you're experimenting with open-source LLMs, Hugging Face provides a vast array of models that can be integrated with AutoGen.
    *   [Hugging Face Models](https://huggingface.co/models)
*   **Agentic AI Research**: Stay updated with the latest research in multi-agent systems, hierarchical agents, and task decomposition for more advanced patterns.
    *   Search for terms like "hierarchical reinforcement learning for agents" or "multi-agent task decomposition" on arXiv or Google Scholar.
